In [ ]:
# pip install pandas numpy scikit-learn
# pip install cdt

In [9]:
import pandas as pd
import openpyxl
import numpy as np
import os

In [1]:
from sklearn.preprocessing import StandardScaler

In [2]:
from cdt.causality.pairwise import ANM, IGCI

No GPU automatically detected. Setting SETTINGS.GPU to 0, and SETTINGS.NJOBS to cpu_count.


In [3]:
FILE_PATH = r'C:\Users\Lenovo\Desktop\Licenta\Non_exp_Data\Burnout\Copy of M1 Burnout Manuscript Data.xlsx'


In [4]:
# Causal Pair 1: Depression (X) -> Work-Related Burnout (Y)
X_COLUMN_NAME = 'Patient Health Questionnaire 9' 
Y_COLUMN_NAME = 'Work Related Burnout Score'

In [5]:
# --- Data Loading and Preprocessing ---

def load_and_prepare_data(filepath, x_col, y_col):
    """Loads data, selects the two variables, and performs standardization."""
    print(f"Loading data from: {filepath}")
    try:
        # Assuming the data is in the first sheet, adjust sheet_name if needed
        data = pd.read_excel(filepath)
    except FileNotFoundError:
        print(f"Error: File not found at {filepath}. Please check your FILE_PATH.")
        return None, None
    except Exception as e:
        print(f"Error loading file: {e}")
        return None, None

    # Check if required columns exist
    if x_col not in data.columns or y_col not in data.columns:
        print(f"Error: One or both columns ('{x_col}', '{y_col}') not found in the dataset.")
        print("Available columns:", data.columns.tolist())
        return None, None

    # Select and clean the data
    df_pair = data[[x_col, y_col]].dropna()
    print(f"Data loaded. Remaining samples after cleaning: {len(df_pair)}")
    
    if len(df_pair) < 50: # Check for minimum size for reliable inference
         print("Warning: Dataset size is very small. Causal inference results may be unstable.")

    # Standardize data: Causal Discovery algorithms often work best on scaled data
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_pair)
    
    # Extract standardized X and Y as NumPy arrays
    X_data = scaled_data[:, 0]
    Y_data = scaled_data[:, 1]

    return X_data, Y_data

In [6]:
# --- Causal Inference Functions ---

def run_anm(X, Y):
    """Runs the Additive Noise Model (ANM) algorithm."""
    print("\n--- Running Additive Noise Model (ANM) ---")
    
    # Initialize ANM (uses the default PyTorch backend)
    anm_model = ANM()

    # The predict method takes a tuple (X, Y) or (Y, X) and returns a score.
    # Score > 0 means X -> Y, Score < 0 means Y -> X.
    # The absolute value indicates confidence/strength.
    
    # CDT predicts a label: 1 (X->Y) or -1 (Y->X)
    # The 'anm_score' function provides the raw fitness score for X->Y
    score_x_to_y = anm_model.anm_score(X.reshape(-1, 1), Y.reshape(-1, 1))
    score_y_to_x = anm_model.anm_score(Y.reshape(-1, 1), X.reshape(-1, 1))

    # The preferred direction is the one with the higher fit score (closer to 1)
    if score_x_to_y < score_y_to_x:
        direction = f"{Y_COLUMN_NAME} -> {X_COLUMN_NAME}"
        confidence = score_y_to_x
    else:
        direction = f"{X_COLUMN_NAME} -> {Y_COLUMN_NAME}"
        confidence = score_x_to_y

    print(f"ANM Fit Score (X->Y): {score_x_to_y:.4f}")
    print(f"ANM Fit Score (Y->X): {score_y_to_x:.4f}")
    print(f"Inferred Causal Direction: {direction} (Confidence: {confidence:.4f})")
    
    # Note: In ANM, the true causal direction should have a better fit (lower independence test score, higher model score depending on implementation details). 
    # Here, we interpret the higher score as better fit.
    return direction


In [7]:
def run_igci(X, Y):
    """Runs the Information Geometric Causal Inference (IGCI) algorithm."""
    print("\n--- Running Information Geometric Causal Inference (IGCI) ---")
    
    # Initialize IGCI
    igci_model = IGCI()

    # IGCI's predict_proba returns a single score: 
    # Score > 0 indicates X -> Y
    # Score < 0 indicates Y -> X
    # The absolute magnitude indicates confidence/strength.
    
    # We pass the data tuple (X, Y)
    igci_score = igci_model.predict_proba((X, Y))

    if igci_score > 0:
        direction = f"{X_COLUMN_NAME} -> {Y_COLUMN_NAME}"
    else:
        direction = f"{Y_COLUMN_NAME} -> {X_COLUMN_NAME}"
        
    print(f"IGCI Score: {igci_score:.4f} (Positive = X->Y, Negative = Y->X)")
    print(f"Inferred Causal Direction: {direction} (Confidence: {abs(igci_score):.4f})")
    
    return direction


In [11]:
# --- Main Execution ---

if __name__ == "__main__":
    # 1. Load and Prepare Data
    X_data, Y_data = load_and_prepare_data(FILE_PATH, X_COLUMN_NAME, Y_COLUMN_NAME)

    if X_data is None:
        print("\nExiting due to data loading error. Please check configuration.")
    else:
        # 2. Run ANM Algorithm
        anm_result = run_anm(X_data, Y_data)

        # 3. Run IGCI Algorithm
        igci_result = run_igci(X_data, Y_data)
        
        print("\n--- Summary of Results ---")
        print(f"ANM Inferred Direction: {anm_result}")
        print(f"IGCI Inferred Direction: {igci_result}")

Loading data from: C:\Users\Lenovo\Desktop\Licenta\Non_exp_Data\Burnout\Copy of M1 Burnout Manuscript Data.xlsx
Data loaded. Remaining samples after cleaning: 147

--- Running Additive Noise Model (ANM) ---
ANM Fit Score (X->Y): 6.3754
ANM Fit Score (Y->X): 6.5020
Inferred Causal Direction: Work Related Burnout Score -> Patient Health Questionnaire 9 (Confidence: 6.5020)

--- Running Information Geometric Causal Inference (IGCI) ---
IGCI Score: -0.0812 (Positive = X->Y, Negative = Y->X)
Inferred Causal Direction: Work Related Burnout Score -> Patient Health Questionnaire 9 (Confidence: 0.0812)

--- Summary of Results ---
ANM Inferred Direction: Work Related Burnout Score -> Patient Health Questionnaire 9
IGCI Inferred Direction: Work Related Burnout Score -> Patient Health Questionnaire 9
